## Azure ML Data Stores and Data Sets

### Set up the Azure ML Client

In [ ]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import AccountKeyConfiguration
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

print(f"Ready to use Azure ML SDK v2 to work with {ml_client.workspace_name}")

### View all Default Datastores in your workspace

In [ ]:
# Get the default datastore
default_ds = ml_client.datastores.get_default()

# List all datastores and check which one is default
for ds in ml_client.datastores.list():
    print(ds.name, "- Default =", ds.name == default_ds.name)

### Connect a new Azure Blob Storage account as a Datastore

In [ ]:
aml_datastore = AzureBlobDatastore(
    name="aml_datastore",
    description="Datastore pointing to a blob container using https protocol.",
    account_name="YOUR-STORAGE-ACCOUNT-NAME",
    container_name="YOUR-CONTAINER-NAME",
    protocol="https",
    credentials=AccountKeyConfiguration(
        account_key="YOUR-ACCOUNT-KEY"
    ),
)

ml_client.create_or_update(aml_datastore)

### Getting the Default Datastore

In [ ]:
# Get the default datastore
default_ds = ml_client.datastores.get_default()

# Enumerate all datastores and indicate which one is the default
for ds in ml_client.datastores.list():
    print(ds.name, "- Default =", ds.name == default_ds.name)

### Upload data to your Datastore and create a Data Asset

In [ ]:
import requests
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# Upload file to datastore path
from azure.ai.ml.operations import DatastoreOperations

datastore = ml_client.datastores.get_default()

# Construct the datastore path
datastore_path = f"azureml://datastores/{datastore.name}/paths/diabetes.csv"

# Register the uploaded file as a data asset
data_asset = Data(
    path=datastore_path,
    type=AssetTypes.URI_FILE,
    name="diabetes-data",
    description="Diabetes dataset uploaded to datastore"
)

ml_client.data.create_or_update(data_asset)

In [ ]:
import mltable
from mltable import MLTableHeaders, MLTableFileEncoding
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# Get datastore
datastore = ml_client.datastores.get_default()

#constructing File URI
file_uri = (
    f"azureml://subscriptions/{ml_client.subscription_id}"
    f"/resourcegroups/{ml_client.resource_group_name}"
    f"/workspaces/{ml_client.workspace_name}"
    f"/datastores/{datastore.name}"
    f"/paths/diabetes.csv"
)

# Path to CSV in datastore
paths = [{
    "file": file_uri
}]

# Create MLTable from CSV
tbl = mltable.from_delimited_files(
    paths=paths,
    delimiter=",",
    header=MLTableHeaders.all_files_same_headers,
    infer_column_types=True,
    encoding=MLTableFileEncoding.utf8
)

# Preview dataset
print(tbl.show())

# Save MLTable definition locally
mltable_folder = "./diabetes_mltable"
tbl.save(mltable_folder)

# Register MLTable as data asset
data_asset = Data(
    path=mltable_folder,
    type=AssetTypes.MLTABLE,
    name="diabetes-mltable",
    description="Diabetes dataset stored as MLTable"
)

ml_client.data.create_or_update(data_asset)

In [ ]:
print("Datasets:")

for data_asset in ml_client.data.list():
    print("\t", data_asset.name, "version", data_asset.version)